In [10]:
# --- 0. Install Dependencies & Load Local HuggingFace Model (No API Key) ---
!pip install -q dspy transformers accelerate torch

import torch
from transformers import pipeline
from typing import Literal
import dspy
from dspy.lm15 import Message, Response, Usage
from dspy.evaluate import Evaluate
from dspy.teleprompt import BootstrapFewShotWithRandomSearch

# Load Qwen2.5-3B-Instruct onto Colab's T4 GPU
hf_pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-3B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto",
)

# Custom DSPy 3.5 Engine wrapping the HuggingFace pipeline
class HFEngine:
    def __init__(self, pipe):
        self.pipe = pipe

    def complete(self, request):
        messages = []
        if getattr(request, "system", None):
            messages.append({"role": "system", "content": request.system})
        for msg in request.messages:
            text = getattr(msg, "text", None) or "".join(
                getattr(p, "text", str(p)) for p in getattr(msg, "parts", [])
            )
            messages.append({"role": msg.role, "content": text})

        out = self.pipe(
            messages,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=self.pipe.tokenizer.eos_token_id,
        )
        first_item = next(iter(out))
        reply_text = first_item["generated_text"][-1]["content"]

        return Response(
            id=None,
            model=request.model,
            message=Message.assistant(reply_text),
            finish_reason="stop",
            usage=Usage(),
        )

lm = dspy.LM("custom/qwen2.5-3b", engine=HFEngine(hf_pipe))
dspy.configure(lm=lm)
# --------------------------------------------------------------------------

# 1. Dataset (10 train examples + 8 unseen test examples)
data = [
    # Trainset (10 examples)
    ("Enterprise client Acme Corp reports production API is down 500 errors.", "SRE_INCIDENT", "P0"),
    ("I was overcharged 45 dollars on my starter plan last week.", "STANDARD_BILLING", "P2"),
    ("We noticed an exposed S3 bucket link in your public docs.", "SECOPS", "P0"),
    ("Duplicate wire charge of 1200 dollars on our annual invoice.", "VIP_FINANCE", "P1"),
    ("Can you add dark mode to the analytics page?", "PRODUCT_BACKLOG", "P3"),
    ("Our enterprise SSO login is completely broken for 400 employees.", "SRE_INCIDENT", "P0"),
    ("Need a refund for a 19 dollar accidental renewal.", "STANDARD_BILLING", "P2"),
    ("Unauthorized login attempt detected from unknown IP on admin account.", "SECOPS", "P0"),
    ("Invoice discrepancy of 850 dollars on our Q3 seat expansion.", "VIP_FINANCE", "P1"),
    ("Would love an export-to-Notion button on reports.", "PRODUCT_BACKLOG", "P3"),

    # Unseen Test Set (8 examples)
    ("GlobalTech (Enterprise) checkout service is timing out in production.", "SRE_INCIDENT", "P0"),
    ("Charged 650 dollars twice for our team upgrade yesterday.", "VIP_FINANCE", "P1"),
    ("My 15 dollar monthly receipt has the wrong company address.", "STANDARD_BILLING", "P2"),
    ("Found an XSS vulnerability in the profile bio field.", "SECOPS", "P0"),
    ("Please add keyboard shortcuts for navigating tickets.", "PRODUCT_BACKLOG", "P3"),
    ("Enterprise webhook delivery has been stalled for 2 hours.", "SRE_INCIDENT", "P0"),
    ("Double billed 99 dollars on my personal pro subscription.", "STANDARD_BILLING", "P2"),
    ("Overbilled by 2400 dollars on our yearly contract renewal.", "VIP_FINANCE", "P1"),
]

dataset = [
    dspy.Example(ticket=t, queue=q, priority=p).with_inputs("ticket")
    for t, q, p in data
]
trainset, testset = dataset[:10], dataset[10:]

def exact_routing_metric(example, pred, trace=None):
    return (
        example.queue.strip().upper() == pred.queue.strip().upper()
        and example.priority.strip().upper() == pred.priority.strip().upper()
    )

# 2. Fair Baseline: Typical Manual Prompt (without domain calibration)
class ManualPromptRouter(dspy.Module):
    def forward(self, ticket):
        prompt = (
            "Route this support ticket. Reply in exact format:\n"
            "QUEUE: <queue_name>\nPRIORITY: <P0-P3>\n\n"
            f"Ticket: {ticket}"
        )
        raw = str(next(iter(lm(prompt))))
        queue, priority = "", ""
        for line in raw.splitlines():
            if line.upper().startswith("QUEUE:"):
                _, val = line.split(":", 1)
                queue = val.strip()
            elif line.upper().startswith("PRIORITY:"):
                _, val = line.split(":", 1)
                priority = val.strip()
        return dspy.Prediction(queue=queue, priority=priority)

# 3. Typed DSPy Signature (Constrains output domain & specifies SLA routing rules)
QueueType = Literal["SRE_INCIDENT", "VIP_FINANCE", "STANDARD_BILLING", "SECOPS", "PRODUCT_BACKLOG"]
PriorityType = Literal["P0", "P1", "P2", "P3"]

class RouteTicket(dspy.Signature):
    """Route support tickets using internal SLA rules: enterprise outages -> SRE_INCIDENT (P0),
    security issues -> SECOPS (P0), billing >= 500 dollars -> VIP_FINANCE (P1),
    billing < 500 dollars -> STANDARD_BILLING (P2), feature requests -> PRODUCT_BACKLOG (P3)."""
    ticket: str = dspy.InputField()
    queue: QueueType = dspy.OutputField()
    priority: PriorityType = dspy.OutputField()

class DSPyRouter(dspy.Module):
    def __init__(self):
        super().__init__()
        self.route = dspy.Predict(RouteTicket)

    def forward(self, ticket):
        return self.route(ticket=ticket)

# Sanity check on first training example
first_example = next(iter(trainset))
test_router = DSPyRouter()
print("Sanity Check:", test_router(ticket=first_example.ticket))

# 4. Optimize prompt demonstrations using metric-driven search
optimizer = BootstrapFewShotWithRandomSearch(
    metric=exact_routing_metric,
    max_bootstrapped_demos=4,
    max_labeled_demos=8,
    num_candidate_programs=2,
    num_threads=1,
    max_errors=0,
)
optimized_router = optimizer.compile(DSPyRouter(), trainset=trainset)

# 5. Evaluate & Compare
evaluator = Evaluate(devset=testset, metric=exact_routing_metric, num_threads=1, display_progress=False)

manual_result = evaluator(ManualPromptRouter())
dspy_result = evaluator(optimized_router)

print(f"\n--- FINAL UNSEEN TEST SET RESULTS ---")
print(f"Manual User Prompt Accuracy: {manual_result.score}%")
print(f"DSPy Optimized Accuracy:     {dspy_result.score}%")

# 6. Side-by-Side Comparison Table on Unseen Test Cases
print("\n--- SAMPLE PREDICTIONS ---")
for ex in testset[:4]:
    m_pred = ManualPromptRouter()(ex.ticket)
    d_pred = optimized_router(ex.ticket)
    print(f"Ticket:   {ex.ticket}")
    print(f"Expected: {ex.queue} ({ex.priority}) | Manual: {m_pred.queue} ({m_pred.priority}) | DSPy: {d_pred.queue} ({d_pred.priority})\n")


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Sanity Check: Prediction(
    queue='SRE_INCIDENT',
    priority='P0'
)
Going to sample between 1 and 4 traces per predictor.
Will attempt to bootstrap 2 candidate sets.
Average Metric: 6.00 / 10 (60.0%): 100%|██████████| 10/10 [00:10<00:00,  1.01s/it]

2026/11/20 03:59:51 INFO dspy.evaluate.evaluate: Average Metric: 6 / 10 (60.0%)



New best score: 60.0 for seed -3
Scores so far: [60.0]
Best score so far: 60.0
Average Metric: 8.00 / 10 (80.0%): 100%|██████████| 10/10 [00:12<00:00,  1.22s/it]

2026/11/20 04:00:03 INFO dspy.evaluate.evaluate: Average Metric: 8 / 10 (80.0%)



New best score: 80.0 for seed -2
Scores so far: [60.0, 80.0]
Best score so far: 80.0


... (10 outputs left out)

2026/11/20 04:01:04 INFO dspy.evaluate.evaluate: Average Metric: 8 / 10 (80.0%)



Scores so far: [60.0, 80.0, 90.0, 90.0, 80.0]
Best score so far: 90.0
5 candidate programs found.


2026/11/20 04:01:14 INFO dspy.evaluate.evaluate: Average Metric: 0 / 8 (0.0%)
2026/11/20 04:01:23 INFO dspy.evaluate.evaluate: Average Metric: 4 / 8 (50.0%)



--- FINAL UNSEEN TEST SET RESULTS ---
Manual User Prompt Accuracy: 0.0%
DSPy Optimized Accuracy:     50.0%

--- SAMPLE PREDICTIONS ---
Ticket:   GlobalTech (Enterprise) checkout service is timing out in production.
Expected: SRE_INCIDENT (P0) | Manual: Production_Support (P2) | DSPy: SRE_INCIDENT (P3)

Ticket:   Charged 650 dollars twice for our team upgrade yesterday.
Expected: VIP_FINANCE (P1) | Manual: Billing_Inquiries (P2) | DSPy: VIP_FINANCE (P1)

Ticket:   My 15 dollar monthly receipt has the wrong company address.
Expected: STANDARD_BILLING (P2) | Manual: Billing (P2) | DSPy: STANDARD_BILLING (P2)

Ticket:   Found an XSS vulnerability in the profile bio field.
Expected: SECOPS (P0) | Manual: Security Vulnerability (P2) | DSPy: SECOPS (P0)

